# `04-analysis.ipynb`: Analysis of scraped and cleaned data.

## Setup

In [101]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from umpd_crime.constants import MONTHS, FBI_THEFT_CATEGORIES
from scipy import stats

from pathlib import Path

PROC_DATA_DIR = Path.cwd().parent / "data" / "processed" 

CUR_TIMESTAMP = time.localtime()
CUR_YEAR = CUR_TIMESTAMP.tm_year
CUR_MONTH = CUR_TIMESTAMP.tm_mon

In [5]:
arrests_and_ci = pd.read_csv(PROC_DATA_DIR / f"merged_arrest_ci_data_2018-{CUR_YEAR}.csv")

In [6]:
disposition_hist = arrests_and_ci['disposition'].value_counts().reset_index()
disposition_hist
disposition_hist['perc'] = disposition_hist['count'] / len(arrests_and_ci['disposition'])
disposition_hist

,disposition,count,perc
0,CBE,6466,0.487228
1,Arrest,3067,0.231105
2,Investigation Pending,1743,0.131339
3,Active/Pending,1417,0.106774
4,Unfounded,240,0.018085
5,Summons Issued,174,0.013111
6,Warrant Issued,124,0.009344
7,Juvenile Arrest,40,0.003014


In [7]:
arrests_and_ci['crime_type'].value_counts().head(15)

crime_type
Injured/Sick Person          2710
Theft                        2146
DWI/DUI                      1343
Assist Other Agency           860
CDS Violation                 635
Other Incident                617
Emergency Petition            529
Vandalism                     504
Suspicious Activity           342
Damage to State Property      261
Trespassing                   184
Burglary                      179
Alcohol Violation             179
Dept Property Damage/Loss     176
Theft from Auto               174
Name: count, dtype: int64

In [78]:
arrests_and_ci['date_occurred'] = pd.to_datetime(arrests_and_ci['date_occurred'])
arrests_and_ci['report_date'] = pd.to_datetime(arrests_and_ci['report_date'])
arrests_and_ci['arrest_date'] = pd.to_datetime(arrests_and_ci['arrest_date'])
arrests_and_ci['arrest_date_mo'] = arrests_and_ci['arrest_date'].dt.month_name()
arrests_and_ci['report_date_mo'] = arrests_and_ci['report_date'].dt.month_name()
arrests_and_ci['date_occurred_mo'] = arrests_and_ci['date_occurred'].dt.strftime('%b')
arrests_and_ci['date_occurred_mo'] = pd.Categorical(arrests_and_ci['date_occurred_mo'], MONTHS, True)
arrests_and_ci['date_occurred_yr'] = arrests_and_ci['date_occurred'].dt.year
arrests_and_ci['report_date_mo'].value_counts()

report_date_mo
September    1821
October      1750
April        1270
February     1252
March        1244
November     1152
May          1106
August       1015
January       703
July          703
June          629
December      626
Name: count, dtype: int64

In [9]:
arrests_and_ci[arrests_and_ci['arrest_number'].isna() == False]['arrest_date_mo'].value_counts()

arrest_date_mo
August       325
September    325
March        310
May          310
October      308
July         301
April        298
February     292
January      231
November     226
June         198
December     174
Name: count, dtype: int64

In [80]:
arrests_and_ci['date_occurred_yr'].unique()

array([2019, 2018, 2014, 2012, 2017, 2011, 2007, 2002, 2020, 2010, 2021,
       2001, 2022, 1986, 2023, 2003, 2024, 2004, 2025, 2005, 2026, 2016],
      dtype=int32)

In [90]:
arrests_and_ci.query('date_occurred_yr >= 2017')

,Unnamed: 0,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,...,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges,arrest_date_mo,report_date_mo,date_occurred_mo,date_occurred_yr
0,0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,...,NaN,Asian/Pacific Islander,Male,Fireworks Discharge without the required permit,1.0,['Fireworks Discharge without the required per...,January,January,Jan,2019
1,1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,...,NaN,Black,Female,"(Driving, Attempting to drive) veh. while impa...",6.0,"['(Driving, Attempting to drive) veh. while im...",January,January,Jan,2019
2,2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,...,NaN,Asian/Pacific Islander,Male,"(Driving, Attempting to drive) veh. while impa...",2.0,"['(Driving, Attempting to drive) veh. while im...",January,January,Jan,2019
3,3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,January,Dec,2018
4,4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,...,NaN,Black,Male,CDS:Possession Marijuana L/T 10 grams,1.0,['CDS:Possession Marijuana L/T 10 grams'],January,January,Jan,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13266,13266,2026-00289157,2026-10-31 23:56:00,evening,Fall 2026,Fall,2026-10-31 23:56:00,evening,Fall 2026,Fall,...,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3.0,"['(Driving, Attempting to drive) veh. while im...",November,October,Oct,2026
13267,13267,2026-00289161,2026-11-01 00:00:00,night,Fall 2026,Fall,2026-11-01 00:00:00,night,Fall 2026,Fall,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,November,Nov,2026
13268,13268,2026-00289166,2026-11-01 00:08:00,night,Fall 2026,Fall,2026-11-01 00:08:00,night,Fall 2026,Fall,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,November,Nov,2026
13269,13269,2026-00289167,2026-11-01 00:25:00,night,Fall 2026,Fall,2026-11-01 00:25:00,night,Fall 2026,Fall,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,November,Nov,2026


In [117]:
arrests_and_ci = arrests_and_ci.query('date_occurred_yr > 2017')
arrests_and_ci.drop_duplicates('umpd_case_number')

,Unnamed: 0,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,...,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges,arrest_date_mo,report_date_mo,date_occurred_mo,date_occurred_yr
0,0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,...,NaN,Asian/Pacific Islander,Male,Fireworks Discharge without the required permit,1.0,['Fireworks Discharge without the required per...,January,January,Jan,2019
1,1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,...,NaN,Black,Female,"(Driving, Attempting to drive) veh. while impa...",6.0,"['(Driving, Attempting to drive) veh. while im...",January,January,Jan,2019
2,2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,...,NaN,Asian/Pacific Islander,Male,"(Driving, Attempting to drive) veh. while impa...",2.0,"['(Driving, Attempting to drive) veh. while im...",January,January,Jan,2019
3,3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,January,Dec,2018
4,4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,...,NaN,Black,Male,CDS:Possession Marijuana L/T 10 grams,1.0,['CDS:Possession Marijuana L/T 10 grams'],January,January,Jan,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13266,13266,2026-00289157,2026-10-31 23:56:00,evening,Fall 2026,Fall,2026-10-31 23:56:00,evening,Fall 2026,Fall,...,NaN,White,Male,"(Driving, Attempting to drive) veh. while impa...",3.0,"['(Driving, Attempting to drive) veh. while im...",November,October,Oct,2026
13267,13267,2026-00289161,2026-11-01 00:00:00,night,Fall 2026,Fall,2026-11-01 00:00:00,night,Fall 2026,Fall,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,November,Nov,2026
13268,13268,2026-00289166,2026-11-01 00:08:00,night,Fall 2026,Fall,2026-11-01 00:08:00,night,Fall 2026,Fall,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,November,Nov,2026
13269,13269,2026-00289167,2026-11-01 00:25:00,night,Fall 2026,Fall,2026-11-01 00:25:00,night,Fall 2026,Fall,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,November,Nov,2026


In [118]:
arrests_and_ci_all_dt_occurred_yr_mo = arrests_and_ci.value_counts(['date_occurred_yr', 'date_occurred_mo']).reset_index()
arrests_and_ci_all_dt_occurred_yr_mo = arrests_and_ci_all_dt_occurred_yr_mo.sort_values(['date_occurred_yr', 'date_occurred_mo'])
arrests_and_ci_all_dt_occurred_yr_mo['yr-mo'] = arrests_and_ci_all_dt_occurred_yr_mo['date_occurred_yr'].astype('str') + '-' + arrests_and_ci_all_dt_occurred_yr_mo['date_occurred_mo'].astype('str')
arrests_and_ci_all_dt_occurred_yr_mo = arrests_and_ci_all_dt_occurred_yr_mo.sort_values(['date_occurred_yr', 'date_occurred_mo'])

arrests_and_ci_all_dt_occurred_yr_mo
arrests_and_ci_all_dt_occurred_yr_mo['yr-mo'] = pd.Categorical(arrests_and_ci_all_dt_occurred_yr_mo['yr-mo'], arrests_and_ci_all_dt_occurred_yr_mo['yr-mo'].tolist(), True)
arrests_and_ci_all_dt_occurred_yr_mo['yr_mo_dt'] = pd.to_datetime(arrests_and_ci_all_dt_occurred_yr_mo['yr-mo'].astype('str'), format="%Y-%b")

In [119]:
arrests_and_ci_all_dt_occurred_yr_mo.query('date_occurred_mo in ["Aug", "Sep", "Oct"]')

,date_occurred_yr,date_occurred_mo,count,yr-mo,yr_mo_dt
100,2018,Aug,1,2018-Aug,2018-08-01
99,2018,Sep,1,2018-Sep,2018-09-01
96,2018,Oct,2,2018-Oct,2018-10-01
29,2019,Aug,171,2019-Aug,2019-08-01
5,2019,Sep,241,2019-Sep,2019-09-01
18,2019,Oct,203,2019-Oct,2019-10-01
84,2020,Aug,64,2020-Aug,2020-08-01
47,2020,Sep,131,2020-Sep,2020-09-01
52,2020,Oct,119,2020-Oct,2020-10-01
54,2021,Aug,114,2021-Aug,2021-08-01


In [120]:
px.line(
  arrests_and_ci_all_dt_occurred_yr_mo,
  x = 'yr_mo_dt',
  y = 'count'
)

In [121]:
px.scatter(
  arrests_and_ci_all_dt_occurred_yr_mo,
  x = 'yr_mo_dt',
  y = 'count',
  trendline = 'ols'
)

## Uniform Crime Reports

There's the 2023-2026 National Incident-Based Reporting System (NIBRS) and the 2012-2022 Historical Summary Reporting System (SRS).

In [10]:
nibrs = pd.read_csv(PROC_DATA_DIR / "uniform-crime-reports" / f"ucr_2023-{CUR_YEAR}_monthly.csv")

### NIBRS

In [57]:
nibrs['month'] = pd.Categorical(nibrs['month'], categories=MONTHS, ordered=True)
nibrs

,year,crime,crime_type,month,count
0,2025,Robbery,crimes_against_property,Jan,0
1,2025,Arson,crimes_against_property,Jan,0
2,2025,Extortion/Blackmail,crimes_against_property,Jan,0
3,2025,Burglary/Breaking or Entering,crimes_against_property,Jan,0
4,2025,Pocket-picking,crimes_against_property,Jan,0
...,...,...,...,...,...
3019,2026,Betting/Wagering,crimes_against_society,Dec,0
3020,2026,Operating/Promoting/Assisting Gambling,crimes_against_society,Dec,0
3021,2026,Gambling Equipment Violations,crimes_against_society,Dec,0
3022,2026,Sports Tampering,crimes_against_society,Dec,0


#### All incidents

Let's look at some crimes individually

In [65]:
nibrs_all_monthly = nibrs.groupby(['year', 'month'])['count'].sum().reset_index()
nibrs_all_monthly['yr-mo'] = nibrs_all_monthly['year'].astype('str') + "-" + nibrs_all_monthly['month'].astype('str')
nibrs_all_monthly['yr-mo'] = pd.Categorical(nibrs_all_monthly['yr-mo'], nibrs_all_monthly['yr-mo'].tolist(), True)

nibrs_all_monthly = nibrs_all_monthly[nibrs_all_monthly['yr-mo'] <= "2026-Aug"]

nibrs_all_monthly_fig = px.line(nibrs_all_monthly, "yr-mo", "count")
nibrs_all_monthly_fig.show()

#### Driving Under the Influence (DUI)

In [12]:
dui_23_26 = nibrs[nibrs['crime'] == "Driving Under the Influence"].pivot(
  index = 'month',
  columns = 'year',
  values = 'count'
)

dui_23_26_t = dui_23_26.T.reset_index()

dui_23_26[f'2023-{CUR_YEAR}_%_change'] = ((dui_23_26[2026] - dui_23_26[2023]) / dui_23_26[2023]) * 100

In [13]:
dui_23_26

year,2023,2024,2025,2026,2023-2026_%_change
month,,,,,
Jan,5,10,17,10,100.000000
Feb,13,9,11,17,30.769231
Mar,8,9,19,23,187.500000
Apr,7,11,10,25,257.142857
May,11,6,16,40,263.636364
Jun,7,14,9,27,285.714286
Jul,11,26,14,20,81.818182
Aug,10,15,22,42,320.000000
Sep,3,7,11,0,-100.000000


In [14]:
dui_23_26_t

month,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,2023,5,13,8,7,11,7,11,10,3,9,7,6
1,2024,10,9,9,11,6,14,26,15,7,8,10,13
2,2025,17,11,19,10,16,9,14,22,11,16,21,16
3,2026,10,17,23,25,40,27,20,42,0,0,0,0


In [15]:
dui_23_26_t['total'] = dui_23_26_t[MONTHS].sum(axis=1)

In [16]:
dui_23_26_t.T

,0,1,2,3
month,,,,
year,2023,2024,2025,2026
Jan,5,10,17,10
Feb,13,9,11,17
Mar,8,9,19,23
Apr,7,11,10,25
May,11,6,16,40
Jun,7,14,9,27
Jul,11,26,14,20
Aug,10,15,22,42


In [17]:
((204-97)/97) * 100

110.30927835051547

#### Simple Assault

In [18]:
simple_assault = nibrs[nibrs['crime'] == "Simple Assault"].pivot(
  index = "month",
  columns= "year",
  values = 'count'
)

simple_assault_t = simple_assault.T.reset_index()

simple_assault[f"2023-{CUR_YEAR}_%_change"] = ((simple_assault[2026] - simple_assault[2023]) / simple_assault[2023]) * 100

In [19]:
simple_assault

year,2023,2024,2025,2026,2023-2026_%_change
month,,,,,
Jan,0,0,0,0,NaN
Feb,4,1,2,2,-50.000000
Mar,3,1,2,1,-66.666667
Apr,4,0,2,4,0.000000
May,6,2,2,2,-66.666667
Jun,3,2,0,0,-100.000000
Jul,2,1,1,2,0.000000
Aug,2,0,0,1,-50.000000
Sep,4,1,2,0,-100.000000


In [20]:
simple_assault_t['total'] = simple_assault_t[MONTHS].sum(axis=1)

In [21]:
((12-35)/35) * 100

-65.71428571428571

#### Theft

In [55]:
nibrs[nibrs['crime'].isin(FBI_THEFT_CATEGORIES)].groupby('year')['count'].sum()

year
2023    372
2024    498
2025    346
2026    120
Name: count, dtype: int64

In [56]:
monthly_theft_df = nibrs[nibrs['crime'].isin(FBI_THEFT_CATEGORIES)].groupby(['year', 'month'])['count'].sum().reset_index()

monthly_theft_df['year'] = monthly_theft_df['year']

monthly_theft_df['yr-mo'] = monthly_theft_df['year'].astype('str') + "-" + monthly_theft_df['month'].astype('str')

monthly_theft_df['yr-mo'] = pd.Categorical(monthly_theft_df['yr-mo'], monthly_theft_df['yr-mo'].tolist(), ordered=True)

monthly_theft_df = monthly_theft_df[monthly_theft_df['yr-mo'] <= '2026-Aug']

fig = px.line(monthly_theft_df, x='yr-mo', y='count')
fig.show()

##### Theft From Building

In [20]:
theft_from_building = nibrs[nibrs['crime'] == "Theft From Building"].pivot(
  index = 'month',
  columns = 'year',
  values = 'count'
)

theft_from_building_t = theft_from_building.T

theft_from_building

year,2023,2024,2025,2026
month,,,,
Jan,5,5,6,1
Feb,5,11,4,5
Mar,6,9,5,6
Apr,6,14,2,5
May,12,4,10,2
Jun,2,3,2,0
Jul,2,3,8,1
Aug,5,9,10,4
Sep,5,6,9,0


In [21]:
theft_from_building_t['total'] = theft_from_building_t[MONTHS].sum(axis=1)

In [22]:
theft_from_building_t

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,total
year,,,,,,,,,,,,,
2023,5,5,6,6,12,2,2,5,5,4,12,7,71
2024,5,11,9,14,4,3,3,9,6,10,3,7,84
2025,6,4,5,2,10,2,8,10,9,5,4,6,71
2026,1,5,6,5,2,0,1,4,0,0,0,0,24
